# Training (Spark ML, GCP YARN)
Trains LR model on baseline features and Holt-Winters features.
Outputs are saved to HDFS under /user/tiennd3886.

In [1]:
import pandas as pd
from datetime import datetime
from pyspark.sql import SparkSession, Window, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from xgboost.spark import SparkXGBRegressor
from pyspark.ml.evaluation import RegressionEvaluator

BASE_HDFS = "/user/tiennd3886"
FEATURE_PATH = f"{BASE_HDFS}/feature_engineering/demand_prediction_features_30m_enhanced"
OUT_BASE_ROOT = f"{BASE_HDFS}/results/sparkml"
MODEL_BASE = f"{BASE_HDFS}/models/sparkml"
TARGET_COL = "pickup_demand_t1"

feature_cols_base = [
    "hour", "dow", "month", "is_weekend",
    "is_holiday", "is_covid_lockdown", "is_covid_partial",
    "lag_6", "lag_12", "lag_336",
    "roll_mean_12", "roll_mean_48", "roll_std_48", "cluster_id",
]
feature_cols_hw = feature_cols_base + ["hw_forecast"]

spark = (
    SparkSession.builder
    .appName("DemandPredictionFeatureEngineering_GCP")
    .master("yarn")
    .config("spark.submit.deployMode", "client")
    .config("spark.eventLog.enabled", "true")
    .config("spark.executor.instances", "3")
    .config("spark.executor.cores", "3")
    .config("spark.executor.memory", "8g")
    .config("spark.executor.memoryOverhead", "1g")
    .config("spark.driver.memory", "3g")
    .config("spark.driver.memoryOverhead", "1g")
    .config("spark.sql.shuffle.partitions", "64")
    .config("spark.sql.parquet.mergeSchema", "true")
    .config("spark.sql.parquet.enableVectorizedReader", "false")
    .getOrCreate()
 )
spark.sparkContext.setLogLevel("WARN")

df_all = spark.read.parquet(FEATURE_PATH)
train_df = df_all.filter(F.col("split") == "train").cache()
val_df = df_all.filter(F.col("split") == "val").cache()
test_df = df_all.filter(F.col("split") == "test").cache()
print("Train/Val/Test rows:", train_df.count(), val_df.count(), test_df.count())


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/25 14:34:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/25 14:34:35 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


Train/Val/Test rows: 19112736 2624740 5680800


In [2]:
rmse_eval = RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="rmse")
mae_eval = RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="mae")
r2_eval = RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="r2")

def add_smape(df):
    denom = (F.abs(F.col(TARGET_COL)) + F.abs(F.col("prediction")))
    smape = F.when(denom == 0, F.lit(0.0)).otherwise(200.0 * F.abs(F.col(TARGET_COL) - F.col("prediction")) / denom)
    return df.withColumn("smape", smape)

def evaluate(model_name, pred_df):
    pred_df = pred_df.withColumn("prediction", F.when(F.col("prediction") < 0, 0.0).otherwise(F.col("prediction")))
    rmse = float(rmse_eval.evaluate(pred_df))
    mae = float(mae_eval.evaluate(pred_df))
    r2 = float(r2_eval.evaluate(pred_df))
    mape = float(pred_df.agg(F.avg(F.abs(F.col(TARGET_COL) - F.col("prediction")) / F.col(TARGET_COL)) * 100.0).first()[0])
    smape = float(add_smape(pred_df).agg(F.avg(F.col("smape"))).first()[0])
    return {"model": model_name, "RMSE": rmse, "MAE": mae, "MAPE": mape, "sMAPE": smape, "R2": r2}

assembler_base = VectorAssembler(inputCols=feature_cols_base, outputCol="features", handleInvalid="skip")
assembler_hw = VectorAssembler(inputCols=feature_cols_hw, outputCol="features", handleInvalid="skip")

model_setups = {
    "lr_base": {"model": LinearRegression(featuresCol="features", labelCol=TARGET_COL, predictionCol="prediction", maxIter=50), "assembler": assembler_base},
    "lr_hw": {"model": LinearRegression(featuresCol="features", labelCol=TARGET_COL, predictionCol="prediction", maxIter=50), "assembler": assembler_hw}
}

metrics_rows = []
pred_union = None
fitted_models = {}

for name, setup in model_setups.items():
    pipeline = Pipeline(stages=[setup["assembler"], setup["model"]])
    fitted = pipeline.fit(train_df)
    fitted_models[name] = fitted
    val_pred = fitted.transform(val_df)
    test_pred = fitted.transform(test_df)
    val_metrics = evaluate(name + "_val", val_pred)
    test_metrics = evaluate(name + "_test", test_pred)
    metrics_rows.extend([val_metrics, test_metrics])
    slim_pred = test_pred.select(F.lit(name).alias("model"), "PULocationID", "pickup_bin_30m", TARGET_COL, "prediction")
    pred_union = slim_pred if pred_union is None else pred_union.unionByName(slim_pred)
    print("Finished model:", name)


26/05/25 14:36:12 WARN Instrumentation: [eef14d69] regParam is zero, which might cause numerical instability and overfitting.
26/05/25 14:36:29 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/05/25 14:36:29 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK
26/05/25 14:36:49 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Finished model: lr_base


26/05/25 14:37:12 WARN Instrumentation: [6073aef5] regParam is zero, which might cause numerical instability and overfitting.


Finished model: lr_hw


In [3]:
metrics_pdf = pd.DataFrame(metrics_rows)
display(metrics_pdf)

val_rows = metrics_pdf[metrics_pdf["model"].str.endswith("_val")].copy()
best_val = val_rows.sort_values("sMAPE").iloc[0]["model"].replace("_val", "")
print("Best model by val sMAPE:", best_val)

run_id = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
OUT_BASE = f"{OUT_BASE_ROOT}/run_{run_id}"
MODEL_PATH = f"{MODEL_BASE}/run_{run_id}/{best_val}"

metrics_sdf = spark.createDataFrame(metrics_rows)
metrics_sdf.write.mode("overwrite").parquet(f"{OUT_BASE}/metrics")
pred_union.write.mode("overwrite").partitionBy("model").parquet(f"{OUT_BASE}/predictions")
fitted_models[best_val].write().overwrite().save(MODEL_PATH)

print("Saved:")
print("-", f"{OUT_BASE}/metrics")
print("-", f"{OUT_BASE}/predictions")
print("-", MODEL_PATH, "(Spark ML format)")


,model,RMSE,MAE,MAPE,sMAPE,R2
0,lr_base_val,8.441830,2.780905,72.420229,101.901055,0.903277
1,lr_base_test,9.435927,3.288340,71.390379,96.974171,0.902242
2,lr_hw_val,8.520171,2.827056,73.887082,102.513465,0.901474
3,lr_hw_test,11.084150,3.765106,79.219189,98.069975,0.865108


/tmp/ipykernel_45894/1511749127.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  run_id = datetime.utcnow().strftime("%Y%m%d_%H%M%S")


Best model by val sMAPE: lr_base


Saved:
- /user/tiennd3886/results/sparkml/run_20260525_143809/metrics
- /user/tiennd3886/results/sparkml/run_20260525_143809/predictions
- /user/tiennd3886/models/sparkml/run_20260525_143809/lr_base (Spark ML format)


In [4]:
spark.catalog.clearCache()
spark.stop()
